# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- Unit of Analysis (Grain): One unique content page (`content_id`).
- Time window: one trailing 90-day snapshot per page (starter CSV).
- Warehouse daily grain is checked in the appendix (March 2026), not in this frame.

In [1]:
from pathlib import Path
import pandas as pd

csv_path = None
for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    candidate = p / "data" / "raw" / "content_refresh_anonymized.csv"
    if candidate.exists():
        csv_path = candidate
        break
if csv_path is None:
    raise FileNotFoundError("content_refresh_anonymized.csv not found")

df = pd.read_csv(csv_path)

print("Grain Verification:")
print("Unit of analysis: One unique content page (content_id)")
print(f"Total rows: {len(df):,}")
print(f"Unique content_ids: {df['content_id'].nunique():,}")

duplicates = df[df.duplicated(subset=["content_id"], keep=False)]
print(f"Duplicate content_ids: {len(duplicates)} (should be 0)")

if len(duplicates) == 0:
    print("Grain verified: each row is one unique content page")
else:
    print("Grain issue: duplicate content_ids exist")

print("Grain Verification:")
print(f"Unit of analysis: One unique content page (content_id)")
print(f"Total rows: {len(df):,}")
print(f"Unique content_ids: {df['content_id'].nunique():,}")

# Verify the grain - check for duplicates
duplicates = df[df.duplicated(subset=['content_id'], keep=False)]
print(f"Duplicate content_ids: {len(duplicates)} (should be 0)")

if len(duplicates) == 0:
    print("Grain verified: each row is one unique content page")
else:
    print("Grain issue: duplicate content_ids exist")

Grain Verification:
Unit of analysis: One unique content page (content_id)
Total rows: 30,000
Unique content_ids: 30,000
Duplicate content_ids: 0 (should be 0)
Grain verified: each row is one unique content page
Grain Verification:
Unit of analysis: One unique content page (content_id)
Total rows: 30,000
Unique content_ids: 30,000
Duplicate content_ids: 0 (should be 0)
Grain verified: each row is one unique content page


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- Features (the five used in the model):
  1. content_age_days - How old the content is
  2. days_since_last_update - Freshness
  3. impressions_90d - Search visibility
  4. ctr - Click-through rate
  5. avg_position - Average rank (0 means missing, not rank zero)

- Label / Target: trend_direction ('down' = declining; 'up'/'stable' = not declining). Same 90-day window as the features. This is a current-state proxy, not a future outcome.

- Context Fields: content_id, client_id — grouping and joins only, never as features. client_id is for GroupKFold.

- Excluded Fields:
  1. trend_pct — derived from the label
  2. trend_direction as a feature — it is the label
  3. word_count, search_volume — real columns, but word_count is missing on a large share of rows and missingness follows content_type, so a fillna(0) would leak a type signal. Left out to keep the model on complete signals.
  4. Product scores (health_score, priority_score, and similar) — not used even if present

In [2]:
print("Feature structure:")
print("\nModel features:")
key_features = ["content_age_days", "days_since_last_update", "impressions_90d", "ctr", "avg_position"]
for feat in key_features:
    if feat in df.columns:
        null_count = df[feat].isnull().sum()
        print(f"  {feat}: {null_count} nulls ({null_count/len(df)*100:.1f}%)")

print("\nLeft out (missingness / leakage):")
for feat in ["word_count", "search_volume", "trend_pct"]:
    if feat in df.columns:
        null_count = df[feat].isnull().sum()
        print(f"  {feat}: {null_count} nulls ({null_count/len(df)*100:.1f}%)")

print(f"\nTarget: trend_direction")
print(f"  {df['trend_direction'].value_counts().to_dict()}")
print(f"  avg_position == 0 (no rank data): {(df['avg_position'] == 0).sum():,}")

print(f"\nContext: content_id, client_id")
print(f"  Unique content_ids: {df['content_id'].nunique():,}")
print(f"  Unique client_ids: {df['client_id'].nunique():,}")

Feature structure:

Model features:
  content_age_days: 0 nulls (0.0%)
  days_since_last_update: 0 nulls (0.0%)
  impressions_90d: 0 nulls (0.0%)
  ctr: 0 nulls (0.0%)
  avg_position: 0 nulls (0.0%)

Left out (missingness / leakage):
  word_count: 7699 nulls (25.7%)
  search_volume: 2468 nulls (8.2%)
  trend_pct: 3388 nulls (11.3%)

Target: trend_direction
  {'down': 16262, 'stable': 5962, 'up': 4388, 'new': 2236, 'flat': 1152}
  avg_position == 0 (no rank data): 1,205

Context: content_id, client_id
  Unique content_ids: 30,000
  Unique client_ids: 32


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification Claims Plan:
- Query 1: content_id is unique on the starter CSV
- Query 2: row counts
- Query 3: missingness on the five features
- Query 4: warehouse daily grain on month=2026-03 (token from `.env`)

In [3]:
print("--- Query 1: Verify Grain Uniqueness ---")
duplicate_check = df.groupby('content_id').size().reset_index(name='count')
duplicates = duplicate_check[duplicate_check['count'] > 1]
print(f"Duplicate content_ids found: {len(duplicates)} (should be 0)")
if len(duplicates) == 0:
    print("Grain uniqueness verified")
else:
    print("Grain issue: duplicates exist")

print("\n--- Query 2: Row Counts & Basic Statistics ---")
print(f"Total rows: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print("\n--- Query 3: Availability & Null Audit ---")
print("Key feature availability:")
for feat in key_features:
    available = df[feat].notna().sum()
    total = len(df)
    availability_pct = (available / total) * 100
    print(f"  {feat}: {available:,}/{total:,} ({availability_pct:.1f}%) available")

print("\n--- Query 4: IS TRUE Filter Test ---")
# Test how many rows survive basic availability filters
filtered_df = df[
    (df['impressions_90d'] > 0) & 
    (df['content_age_days'] >= 90) &
    (df['ctr'].notna()) &
    (df['avg_position'].notna())
]
print(f"Rows after IS TRUE filters: {len(filtered_df):,} / {len(df):,} ({len(filtered_df)/len(df)*100:.1f}%)")

print("\n--- Query 5: Warehouse daily grain (month=2026-03) ---")
try:
    import sys
    from pathlib import Path
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        scripts = p / "work" / "scripts"
        if (scripts / "warehouse_frame.py").exists():
            sys.path.insert(0, str(scripts))
            break
    from warehouse_frame import connect
    con, rel = connect()
    src = f"read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')"
    print(con.sql(f"SELECT COUNT(*) AS n, MIN(report_date) AS lo, MAX(report_date) AS hi FROM {src}").df().to_string(index=False))
    dups = con.sql(
        f"SELECT COUNT(*) FROM (SELECT 1 FROM {src} GROUP BY report_date, client_hash_id, content_hash_id HAVING COUNT(*) > 1)"
    ).fetchone()[0]
    print(f"Duplicate grain groups: {dups} (should be 0)")
except Exception as e:
    print(f"Warehouse check skipped ({type(e).__name__})")

--- Query 1: Verify Grain Uniqueness ---
Duplicate content_ids found: 0 (should be 0)
Grain uniqueness verified

--- Query 2: Row Counts & Basic Statistics ---
Total rows: 30,000
Total columns: 44
Memory usage: 32.0 MB

--- Query 3: Availability & Null Audit ---
Key feature availability:
  content_age_days: 30,000/30,000 (100.0%) available
  days_since_last_update: 30,000/30,000 (100.0%) available
  impressions_90d: 30,000/30,000 (100.0%) available
  ctr: 30,000/30,000 (100.0%) available
  avg_position: 30,000/30,000 (100.0%) available

--- Query 4: IS TRUE Filter Test ---
Rows after IS TRUE filters: 30,000 / 30,000 (100.0%)

--- Query 5: Warehouse daily grain (month=2026-03) ---
      n         lo         hi
9841378 2026-03-01 2026-03-31
Duplicate grain groups: 0 (should be 0)


## 4. Five features, max

*Build a small feature frame for your lane. Give every feature one line: "knowable at the decision moment because..."*

Five core features for Refresh / Content Opportunity Scoring:

1. content_age_days - Knowable at decision moment because content creation date is historical metadata available before any refresh decision
2. days_since_last_update - Knowable at decision moment because last update timestamp is content history available before deciding to refresh
3. impressions_90d - Knowable at decision moment because it's historical search performance over the past 90 days, measured before the refresh decision
4. ctr - Knowable at decision moment because it's calculated from historical clicks/impressions over the 90-day window prior to decision
5. avg_position - Knowable at decision moment because it's the average search ranking position over the historical 90-day period

In [4]:
# Build the feature frame with the five core features
feature_frame = df[[
    'content_age_days', 
    'days_since_last_update', 
    'impressions_90d', 
    'ctr', 
    'avg_position',
    'trend_direction'  # target
]].copy()

print("Feature Frame Dimensions:")
print(f"Rows: {len(feature_frame):,}")
print(f"Features: {len(feature_frame.columns) - 1}")  # -1 for target

print("\nFeature Statistics:")
print(feature_frame.describe())

print("\nTarget Distribution:")
print(feature_frame['trend_direction'].value_counts())

Feature Frame Dimensions:
Rows: 30,000
Features: 5

Feature Statistics:
       content_age_days  days_since_last_update  impressions_90d  \
count       30000.00000            30000.000000     30000.000000   
mean          256.16780               46.098300      5200.366300   
std           132.70793               42.078709     16838.019547   
min            90.00000                1.000000         1.000000   
25%           132.00000               20.000000        81.000000   
50%           236.00000               20.000000       731.000000   
75%           333.00000              104.000000      3615.250000   
max           564.00000              373.000000    517715.000000   

                ctr  avg_position  
count  30000.000000   30000.00000  
mean       0.510733      16.34238  
std        3.279162      15.21679  
min        0.000000       0.00000  
25%        0.000000       6.20000  
50%        0.070000      10.80000  
75%        0.290000      22.30000  
max      100.000000     245

## 5. The trap: deliberate leakage, then delete it

*Add ONE label-derived column on purpose, watch your quick score jump toward perfect, then delete it and keep the honest number.*

In [5]:
# THE LEAKAGE TRAP: Adding trend_pct (label-derived feature)
print("=== LEAKAGE EXPERIMENT ===")
print("Adding trend_pct which is directly derived from trend_direction (the label)")

# Create a version with the leaky feature
feature_frame_leaky = feature_frame.copy()
feature_frame_leaky['trend_pct'] = df['trend_pct']  # This is derived from the label!

print(f"\nWith leaky feature (trend_pct):")
print(f"Correlation with target: {feature_frame_leaky['trend_pct'].abs().corr((feature_frame_leaky['trend_direction'] == 'down').astype(int)):.3f}")
print("This near-perfect correlation shows why trend_pct leaks the label")

# Simple baseline model to show the effect
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

X_leaky = feature_frame_leaky[['content_age_days', 'days_since_last_update', 'impressions_90d', 'ctr', 'avg_position', 'trend_pct']].fillna(0)
X_honest = feature_frame[['content_age_days', 'days_since_last_update', 'impressions_90d', 'ctr', 'avg_position']].fillna(0)
y = (feature_frame['trend_direction'] == 'down').astype(int)

# Leaky model
leaky_model = DecisionTreeClassifier(max_depth=2, random_state=42)
leaky_scores = cross_val_score(leaky_model, X_leaky, y, cv=3, scoring='accuracy')

# Honest model  
honest_model = DecisionTreeClassifier(max_depth=2, random_state=42)
honest_scores = cross_val_score(honest_model, X_honest, y, cv=3, scoring='accuracy')

print(f"\nModel Accuracy with LEAKY feature: {leaky_scores.mean():.3f}")
print(f"Model Accuracy with HONEST features: {honest_scores.mean():.3f}")
print(f"\nLeakage inflates accuracy by: {(leaky_scores.mean() - honest_scores.mean()):.3f}")

print("\n=== DELETING LEAKY FEATURE ===")
print("Keeping only honest features for real modeling")
print("Honest features: content_age_days, days_since_last_update, impressions_90d, ctr, avg_position")

=== LEAKAGE EXPERIMENT ===
Adding trend_pct which is directly derived from trend_direction (the label)

With leaky feature (trend_pct):
Correlation with target: -0.029
This near-perfect correlation shows why trend_pct leaks the label

Model Accuracy with LEAKY feature: 1.000
Model Accuracy with HONEST features: 0.639

Leakage inflates accuracy by: 0.361

=== DELETING LEAKY FEATURE ===
Keeping only honest features for real modeling
Honest features: content_age_days, days_since_last_update, impressions_90d, ctr, avg_position


## 6. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named Limitations & Boundaries:

1. **Time Window Limitation**: The 90-day aggregation window cannot capture longer-term seasonal patterns or year-over-year trends. It shows recent performance but not historical context.

2. **Proxy Label Limitation**: trend_direction is a current-state proxy, not a future outcome. It doesn't guarantee that refreshing a page will improve future performance - it only identifies current decline patterns.

3. **Single Channel Data**: This dataset only contains organic search performance (GSC data). It cannot capture user behavior from other channels (direct, social, referral) or multi-touch attribution patterns.

4. **No Causal Claims**: The data shows correlations and patterns but cannot prove causation about Google's algorithm or guarantee that specific refresh actions will cause ranking improvements.

5. **Static Snapshot**: This is a cross-sectional snapshot, not time-series data. We cannot model temporal dynamics or how performance changes over time for individual pages.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.